# Tennis Shot Analysis Demo

This public notebook supports two execution modes:

1. a synthetic landmark demo that runs anywhere without private assets
2. an optional MediaPipe BlazePose video branch for real tennis clips

To use the real-video branch, install the optional notebook dependencies, set `USE_MEDIAPIPE_VIDEO = True`, and point `VIDEO_PATH` at a local or Colab-accessible tennis video.

In [ ]:
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import cv2
except ImportError:
    cv2 = None

try:
    import mediapipe as mp
except ImportError:
    mp = None

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
USE_MEDIAPIPE_VIDEO = False
VIDEO_PATH = Path('sample_tennis_video.mp4')
MIN_VISIBILITY = 0.5
MAX_FRAMES = None

POSE_LANDMARK_IDS = {
    'wrist': 16,
    'shoulder': 12,
    'hip': 24,
}


def extract_blazepose_landmarks(video_path, min_visibility=0.5, max_frames=None):
    if mp is None or cv2 is None:
        raise ImportError(
            'The real-video branch requires both mediapipe and opencv-python in the notebook environment.'
        )
    if not video_path.exists():
        raise FileNotFoundError(f'Video file not found: {video_path}')

    capture = cv2.VideoCapture(str(video_path))
    fps = capture.get(cv2.CAP_PROP_FPS) or 30.0
    rows = []
    frame_index = 0

    with mp.solutions.pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as pose:
        while True:
            has_frame, frame = capture.read()
            if not has_frame:
                break
            if max_frames is not None and frame_index >= max_frames:
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = pose.process(rgb_frame)

            if result.pose_landmarks is not None:
                landmarks_result = result.pose_landmarks.landmark
                wrist = landmarks_result[POSE_LANDMARK_IDS['wrist']]
                shoulder = landmarks_result[POSE_LANDMARK_IDS['shoulder']]
                hip = landmarks_result[POSE_LANDMARK_IDS['hip']]

                visibility_values = [
                    getattr(wrist, 'visibility', 1.0),
                    getattr(shoulder, 'visibility', 1.0),
                    getattr(hip, 'visibility', 1.0),
                ]
                if min(visibility_values) >= min_visibility:
                    rows.append(
                        {
                            'frame': frame_index,
                            'time_s': frame_index / fps,
                            'hip_x': hip.x,
                            'hip_y': hip.y,
                            'shoulder_x': shoulder.x,
                            'shoulder_y': shoulder.y,
                            'wrist_x': wrist.x,
                            'wrist_y': wrist.y,
                        }
                    )

            frame_index += 1

    capture.release()

    if not rows:
        raise ValueError(
            'No valid pose frames were extracted. Try a clearer clip, lower MIN_VISIBILITY, or verify the video path.'
        )

    return pd.DataFrame(rows)


if USE_MEDIAPIPE_VIDEO:
    print(f'Real-video mode enabled for: {VIDEO_PATH}')
else:
    warnings.warn(
        'USE_MEDIAPIPE_VIDEO is False, so the notebook will run the public synthetic demo path.',
        stacklevel=1,
    )

In [ ]:
if USE_MEDIAPIPE_VIDEO:
    landmarks = extract_blazepose_landmarks(
        VIDEO_PATH,
        min_visibility=MIN_VISIBILITY,
        max_frames=MAX_FRAMES,
    )
    data_source_label = f'MediaPipe video: {VIDEO_PATH.name}'
else:
    frame_count = 80
    time_seconds = np.linspace(0.0, 1.6, frame_count)

    # Synthetic landmark motion for a forehand-like swing.
    hip_x = np.full(frame_count, 0.50)
    hip_y = np.full(frame_count, 0.78)
    shoulder_x = 0.50 + 0.03 * np.sin(time_seconds * math.pi)
    shoulder_y = np.full(frame_count, 0.42)

    phase = np.linspace(-1.2, 1.2, frame_count)
    wrist_x = 0.30 + 0.42 / (1.0 + np.exp(-5.0 * phase))
    wrist_y = 0.63 - 0.18 * np.exp(-(phase ** 2) * 3.5) + 0.02 * np.sin(time_seconds * 10.0)

    landmarks = pd.DataFrame(
        {
            'frame': np.arange(frame_count),
            'time_s': time_seconds,
            'hip_x': hip_x,
            'hip_y': hip_y,
            'shoulder_x': shoulder_x,
            'shoulder_y': shoulder_y,
            'wrist_x': wrist_x,
            'wrist_y': wrist_y,
        }
    )
    data_source_label = 'Synthetic demo sequence'

frame_count = len(landmarks)
landmarks.head()

In [ ]:
dt = landmarks['time_s'].diff().fillna(landmarks['time_s'].iloc[1] - landmarks['time_s'].iloc[0])
dx = landmarks['wrist_x'].diff().fillna(0.0)
dy = landmarks['wrist_y'].diff().fillna(0.0)

landmarks['wrist_speed'] = np.sqrt(dx ** 2 + dy ** 2) / dt
landmarks['torso_rotation_proxy'] = landmarks['shoulder_x'] - landmarks['hip_x']

impact_frame = int(landmarks['wrist_speed'].idxmax())
impact_time = float(landmarks.loc[impact_frame, 'time_s'])
peak_speed = float(landmarks.loc[impact_frame, 'wrist_speed'])

summary = pd.Series(
    {
        'frame_count': frame_count,
        'impact_frame': impact_frame,
        'impact_time_s': round(impact_time, 3),
        'peak_wrist_speed_norm_units': round(peak_speed, 3),
        'mean_torso_rotation_proxy': round(float(landmarks['torso_rotation_proxy'].mean()), 4),
    }
)

summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(landmarks['wrist_x'], landmarks['wrist_y'], color='#c44e52', linewidth=2)
axes[0].scatter(
    landmarks.loc[impact_frame, 'wrist_x'],
    landmarks.loc[impact_frame, 'wrist_y'],
    color='#4c72b0',
    s=80,
    label='Peak speed frame',
)
axes[0].invert_yaxis()
axes[0].set_title(f'{data_source_label}: wrist trajectory')
axes[0].set_xlabel('Normalized x')
axes[0].set_ylabel('Normalized y')
axes[0].legend()

axes[1].plot(landmarks['time_s'], landmarks['wrist_speed'], color='#55a868', linewidth=2)
axes[1].axvline(impact_time, color='#4c72b0', linestyle='--', label='Impact proxy')
axes[1].set_title(f'{data_source_label}: wrist speed over time')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Speed (normalized units / s)')
axes[1].legend()

plt.tight_layout()
plt.show()

summary.to_frame(name='value')